# LangChain: Multimodal Messages

## Outline
* ارسال تصویر به LLM (Image Input)
* تحلیل تصویر با URL و Base64
* ترکیب متن و تصویر در یک پیام
* مثال کاربردی: تحلیل رسید خرید فارسی
* مثال کاربردی: توضیح نمودار و چارت

In [1]:
# pip install langchain langchain-openai python-dotenv requests
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

In [3]:
from langchain.chat_models import init_chat_model

# برای multimodal نیاز به مدل vision-capable داریم
llm = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0)
print("مدل آماده است!")

مدل آماده است!


## ۱. ارسال تصویر با URL

ساده‌ترین روش: آدرس URL تصویر را مستقیم به مدل بدهید.

In [7]:
from langchain.messages import HumanMessage

# ارسال تصویر از طریق URL
message = HumanMessage(content=[
    {
        "type": "image_url",
        "image_url": {
            "url": "https://upload.wikimedia.org/wikipedia/commons/2/23/Azadi_Tower_%2829358497718%29.jpg"
        }
    },
    {
        "type": "text",
        "text": "این تصویر را توصیف کن. به فارسی پاسخ بده."
    }
])

response = llm.invoke([message])
print(response.content)

این تصویر نمایی از برج آزادی در تهران را نشان می‌دهد. برج آزادی با طراحی خاص و مدرن خود، یکی از نمادهای معروف ایران است. در این تصویر، آسمان آبی و صاف دیده می‌شود و برج با خطوط عمودی و منحنی‌های زیبا به وضوح قابل مشاهده است. در اطراف برج، فضای سبز و مسیرهای پیاده‌روی وجود دارد و تعدادی از افراد در حال قدم زدن هستند. در پایین تصویر، یک حوض آب و سنگ‌های سفید دیده می‌شود که به زیبایی محیط افزوده است.


## ۲. ارسال تصویر با Base64



In [28]:
import base64
import requests
from PIL import Image
import io

def url_to_base64(url: str) -> tuple[str, str]:
    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    response = requests.get(url, headers=headers, timeout=20)
    response.raise_for_status()

    # بررسی واقعی تصویر
    try:
        img = Image.open(io.BytesIO(response.content))
        img.verify()  # فقط validate
        format_ = img.format.lower()
    except Exception:
        raise ValueError("Downloaded content is not a valid image")

    media_type = f"image/{format_}"

    image_data = base64.b64encode(response.content).decode("utf-8")

    return image_data, media_type

# دانلود یک تصویر نمونه
image_url = "https://upload.wikimedia.org/wikipedia/commons/2/23/Azadi_Tower_%2829358497718%29.jpg"
image_data, media_type = url_to_base64(image_url)
print(f"نوع تصویر: {media_type}")
print(f"اندازه base64: {len(image_data)} کاراکتر")

نوع تصویر: image/jpeg
اندازه base64: 2428212 کاراکتر


In [30]:
# ارسال تصویر base64 به مدل
message_b64 = HumanMessage(content=[
    {
        "type": "image_url",
        "image_url": {
            "url": f"data:{media_type};base64,{image_data}"
        }
    },
    {
        "type": "text",
        "text": "چه ساختمان‌ها یا مکان‌هایی در این تصویر می‌بینی؟"
    }
])

response = llm.invoke([message_b64])
print(response.content)

در این تصویر، بنای معروف "برج آزادی" (آزادی) در تهران دیده می‌شود. این بنا به عنوان نماد شهر تهران شناخته می‌شود و معماری خاص و زیبایی دارد. همچنین، فضای سبز و مسیرهای پیاده‌روی اطراف آن نیز قابل مشاهده است.


## ۳. تحلیل تصویر از فایل لوکال

اگر تصویر روی کامپیوترتان است:

In [32]:
def image_file_to_base64(file_path: str) -> tuple[str, str]:
    """خواندن تصویر از فایل و تبدیل به base64"""
    import mimetypes
    media_type, _ = mimetypes.guess_type(file_path)
    if media_type is None:
        media_type = "image/jpeg"
    
    with open(file_path, "rb") as f:
        image_data = base64.b64encode(f.read()).decode('utf-8')
    
    return image_data, media_type

def analyze_image(file_path: str, question: str) -> str:
    """تحلیل یک تصویر لوکال"""
    image_data, media_type = image_file_to_base64(file_path)
    
    message = HumanMessage(content=[
        {
            "type": "image_url",
            "image_url": {"url": f"data:{media_type};base64,{image_data}"}
        },
        {"type": "text", "text": question}
    ])
    
    response = llm.invoke([message])
    return response.content

result = analyze_image("images/password.jpg", "متن روی این رسید را بخوان و پسورد را بگو")
print(result)


پسورد روی این رسید "aaeu" است.


## ۴. مثال کاربردی: چند تصویر در یک پیام

مقایسه دو تصویر با هم:

In [40]:
from langchain_core.messages import HumanMessage

# تهران
tehran = "images/Azadi.jpg"
tehran_data, tehran_type = image_file_to_base64(tehran)

# اصفهان
isfahan = "images/isfahan.jpg"
isfahan_data, isfahan_type = image_file_to_base64(isfahan)

# پیام چند تصویری
message_multi = HumanMessage(content=[
    {
        "type": "text",
        "text": "این تصویر اول مربوط به تهران است:"
    },
    {
        "type": "image_url",
        "image_url": {
            "url": f"data:{tehran_type};base64,{tehran_data}"
        }
    },
    {
        "type": "text",
        "text": "این تصویر دوم مربوط به اصفهان است:"
    },
    {
        "type": "image_url",
        "image_url": {
            "url": f"data:{isfahan_type};base64,{isfahan_data}"
        }
    },
    {
        "type": "text",
        "text": "این دو تصویر را مقایسه کن و تفاوت‌های معماری، بافت شهری و فضای فرهنگی را در ۴ جمله به صورت bullet point فارسی توضیح بده."
    }
])

response = llm.invoke([message_multi])
print(response.content)

- **معماری**: تصویر اول نمایی از برج آزادی در تهران را نشان می‌دهد که با طراحی مدرن و خطوط هندسی مشخص می‌شود، در حالی که تصویر دوم نمایی از مسجدی در اصفهان است که با تزئینات سنتی و کاشی‌کاری‌های رنگارنگ شناخته می‌شود.

- **بافت شهری**: تهران به عنوان پایتخت، با ساختمان‌های بلند و مدرن و فضای شهری شلوغ شناخته می‌شود، در حالی که اصفهان با بافت تاریخی و آرامش‌بخش خود، به عنوان یک شهر فرهنگی و گردشگری مشهور است.

- **فضای فرهنگی**: برج آزادی نماد ملی و تاریخی ایران است و بیشتر به عنوان یک نماد مدرن شناخته می‌شود، در حالی که مسجد اصفهان نمایانگر فرهنگ اسلامی و هنرهای سنتی ایرانی است و به عنوان یک مکان مذهبی و فرهنگی اهمیت دارد.

- **تأثیرات اجتماعی**: فضای عمومی در تهران بیشتر به سمت زندگی شهری و مدرن متمایل است، در حالی که اصفهان با فضاهای باز و تاریخی خود، به تعاملات اجتماعی و فرهنگی عمیق‌تری دامن می‌زند.


## ۵. ساخت Agent با قابلیت تحلیل تصویر

یک agent که می‌تواند تصویر بگیرد و آن را تحلیل کند:

In [42]:
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def analyze_image_from_url(url_and_question: str) -> str:
    """
    تصویر را از URL دریافت کرده و تحلیل می‌کند.
    ورودی باید به شکل 'URL|||سوال' باشد.
    مثال: 'https://example.com/img.jpg|||این تصویر چیست؟'
    """
    if '|||' not in url_and_question:
        return "فرمت اشتباه. باید URL|||سوال باشد"
    
    url, question = url_and_question.split('|||', 1)
    url = url.strip()
    question = question.strip()
    
    message = HumanMessage(content=[
        {"type": "image_url", "image_url": {"url": url}},
        {"type": "text", "text": question}
    ])
    response = llm.invoke([message])
    return response.content

# ساخت agent
vision_agent = create_agent(
    model=llm,
    tools=[analyze_image_from_url],
    system_prompt="شما یک دستیار تحلیل تصویر هستید. برای تحلیل تصاویر از ابزار analyze_image_from_url استفاده کنید."
)

print("Vision Agent آماده است!")

Vision Agent آماده است!


In [44]:
# تست agent
response = vision_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "این تصویر از تهران را تحلیل کن: https://upload.wikimedia.org/wikipedia/commons/2/23/Azadi_Tower_%2829358497718%29.jpg"
    }]
})
print(response["messages"][-1].content)

این تصویر نمایی از برج آزادی (آزادی) در تهران، ایران است. برج آزادی به عنوان نماد شهر تهران شناخته می‌شود و در سال 1971 به مناسبت جشن‌های 2500 ساله شاهنشاهی ایران ساخته شده است. طراحی آن ترکیبی از معماری سنتی ایرانی و مدرن است.
